Comenzaremos con la preparación del dataset previa a la implementación del modelo. El primer paso será la importación de las librerías necesarias y el dataset objetivo.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

dataset = 'PlantaA_original.csv' #Lorca_original.csv para el de Lorca, Murcia
                             #PlantaA_original.csv para el de emplazamiento A anonimizado
data = pd.read_csv(dataset, sep=";")

data.head() #Mostramos primeras filas

- datetime: Fecha
- Power_gen: Potencia generada en kilovatios
- temperature: Temperatura en grados celsius
- windSpeed: Velocidad del viento en kilómetros/hora
- solarRad: Irradiancia solar en vatios/metro cuadrado
- humedad: Porcentaje de humedad
- rain: Lluvia en milímetros
- pressure: Presión en hectopascales

Visualizamos información referente al dataset.

In [ ]:
data.info(verbose=True)

Como necesitaremos las columnas en valores numéricos, debemos pasar las de tipo objeto a tipo float antes de continuar.

In [ ]:
cols = [c for c in data.columns if c != "datetime"]

#Antes de pasar a float, limpiamos los datos (eliminando las unidades y dejando solo el valor numérico)
for c in cols:
    s = data[c].astype(str)
    s = s.str.replace(r"[^0-9\.,-]", "", regex=True)
    s = s.str.replace(",", ".", regex=False)
    data[c] = pd.to_numeric(s, errors="coerce")

data.info(verbose=True)

Ahora que tenemos los datos limpios y en formato numérico, mostramos la descripción estadística para observar valores como la media, máximo, ...

In [ ]:
data.describe()

Observamos que hay valores claramente erróneos que alteran los resultados (temperature=1803, windSpeed=410,...). Posteriormente los trataremos.

Histograma:

In [ ]:
histograma = data.hist(figsize=(15, 10))

Podemos observar que los valores de todas las variables se concentran mayoritariamente en una parte del histograma, con valores extraordinarios muy alejados de esta. Power_gen y pressure son la excepción, ya que si presentan histogramas razonables.

Con violinplots se observa claramente:

In [ ]:
#Creamos una lista con las variables numéricas (todas menos datetime)
numeric_cols = data.drop(columns=['datetime'])

#Numero de filas y columnas para los subplots
filas, columnas =4,2

#Creamos figura y ejes para los subplots
fig,axes = plt.subplots(filas, columnas, figsize=(20, 40))
axes = axes.flatten()

for i, col in enumerate(numeric_cols.columns):
    sns.violinplot(y=numeric_cols[col], ax=axes[i])

for ax, col in zip(axes, numeric_cols):
    ax.set_title(f'Distribución de {col}')


plt.tight_layout()
plt.show()

Como los outliers alteran enormemente el resultado del modelo, hemos de eliminarlos del dataset.

Para calcular los outliers, hemos de calcular primero el IQR de cada variable. Esto es la diferencia entre el percentil 75 y el percentil 25, de forma que el valor resultante representa el rango en el que se encuentra el 50% central de los datos. Normalmente, se consideran outliers los valores por debajo de (percentil 25 - 1.5 * IQR) o por encima de (percentil 75 + 1.5 * IQR).

In [ ]:
outlier_cols = data.select_dtypes(include=[np.number]).columns.drop('Power_gen')

P25 = data[outlier_cols].quantile(0.25) #Percentil 25
P75 = data[outlier_cols].quantile(0.75) #Percentil 75
IQR = P75 - P25 #Interquartile Range


#Identificar outliers
outliers = ((data[outlier_cols] < (P25 - 1.5 * IQR)) | (data[outlier_cols] > (P75 + 1.5 * IQR)))
print('Outliers detectados:\n')
print(outliers.sum())

Con los outliers detectados, procedemos a eliminarlos del dataset.

In [ ]:
data = data.loc[~outliers.any(axis=1)].reset_index(drop=True)

Volvemos a mostrar los histogramas y violinplots para ver el resultado

In [ ]:
histograma = data.hist(figsize=(15, 10))

numeric_cols = data.drop(columns=['datetime'])

filas, columnas =4,2

fig,axes = plt.subplots(filas, columnas, figsize=(20, 40))
axes = axes.flatten()

for i, col in enumerate(numeric_cols.columns):
    sns.violinplot(y=numeric_cols[col], ax=axes[i])

for ax, col in zip(axes, numeric_cols):
    ax.set_title(f'Distribución de {col}')


plt.tight_layout()
plt.show()

Consultamos la nueva información y descripción analítica.

In [ ]:
data.info(verbose=True)

In [ ]:
data.describe()

Observamos que los valores imposibles se han eliminado, dejando como resultado los valores reales.

Eliminamos filas con NaN

In [ ]:
data = data.dropna()

Información y descripción del dataset final

In [ ]:
data.info(verbose=True)

In [ ]:
data.describe()

Antes de interpolación

In [ ]:
data['Power_gen'].iloc[33600:33610].plot(kind='line', marker='o')
plt.title(f'Evolución de Power_gen desde índice 33600 hasta 33610')
plt.xlabel('Índice')
plt.ylabel('Valor')
plt.grid(True)
plt.ylim(0, 550)
plt.show()

Interpolacion en muestras con error de medida (0 falso)

In [ ]:
mean = data['Power_gen'].rolling(window=10, center=True).mean()

umbral = data['Power_gen'].max() * 0.05

filtro_error = (data['Power_gen'] == 0) & (mean > umbral)

data.loc[filtro_error, 'Power_gen'] = np.nan 

data['Power_gen'] = data['Power_gen'].interpolate(method='linear')
data['Power_gen'] = data['Power_gen'].fillna(method='ffill').fillna(method='bfill')

Después de interpolación

In [ ]:
data['Power_gen'].iloc[33600:33610].plot(kind='line', marker='o')
plt.title(f'Evolución de Power_gen desde índice 33600 hasta 33610')
plt.xlabel('Índice')
plt.ylabel('Valor')
plt.grid(True)
plt.ylim(0, 550)
plt.show()

Guardamos en un CSV nuevo

In [ ]:
if dataset == 'Lorca_original.csv':
    data = data.loc[data['Power_gen'].shift() != data['Power_gen']]#eliminamos filas con valores repetidos en power_gen de forma local,
                                                                         #ya que la medicion de power_gen no es sincrona con las demas variables
    data.to_csv("Lorca_processed.csv", sep=";", index=False)
elif dataset == 'PlantaA_original.csv':
    data.to_csv("PlantaA_processed.csv", sep=";", index=False)